# 01 · Een sensor kiezen, verbinden en uitlezen (Dragino via Antratek)

**Casus:** Een varkenshouder wil in drie stalafdelingen elke 10 minuten temperatuur en relatieve vochtigheid meten. De sensoren werken op batterij (minstens 5 jaar onderhoudsvrij) en sturen hun gegevens door naar een telefoon. Er is geen netwerkkabel in de stal, wel wifi en 4G op het erf.

In dit notebook doorlopen we de stappen uit de praktijk:

1. Een sensor selecteren bij distributeur **Antratek**;
2. De benodigde instellingen opzoeken in de **handleiding van de fabrikant (Dragino)**;
3. De **ruwe bytes** van een LoRaWAN-bericht handmatig decoderen en controleren;
4. De **NB-IoT-variant** bekijken en een geschikte netwerkprotocolkeuze maken (CoAP, UDP, MQTT, TCP).

> Alle webadressen en sectienummers zijn gecontroleerd op 19 oktober 2026.

## Stap 1 · De webshop: Antratek

Open <https://www.antratek.be/lora/sensor-nodes/temperature-humidity> (*LoRa ▸ Sensor nodes ▸ Temperatuur & vochtigheid*). Hier vind je verschillende LoRaWAN-sensoren (hoofdzakelijk van **Dragino**).

Bekijk de **LSN50v2-S31** (of de nieuwere **S31-LB**). Lees de productbeschrijving en let op de volgende eigenschappen:

| Eigenschap | Waarde / Details | Belang voor de toepassing |
|---|---|---|
| **Artikelnummer** | `LSN50v2-S31-EU868` | **EU868** is de Europese LoRaWAN-frequentieband. |
| **Meetsensor** | Sensirion SHT31 | Industriële, gekalibreerde sensor (±0,2 °C, ±2 % RV, zie handleiding §1.3). |
| **Voeding** | 8500 mAh Li-SOCl2 (tot 10 jaar) | Voldoet aan de eis van 5 jaar; de levensduur hangt af van het zendinterval. |
| **Netwerksleutels** | OTAA (unieke sleutels op het toestel) | Vereist een **LoRaWAN-netwerkserver** (bv. The Things Network) en een gateway. |
| **Prijs** | € 52,95 excl. btw | Voor drie afdelingen bedragen de sensorkosten ± € 160. |
| **Variant** | S31-LB (batterij) / S31-LS (zonnepaneel) | -LB gebruikt een batterij, -LS een zonnepaneel. |

**Netwerkvoorziening:**
Een LoRaWAN-sensor verzendt gegevens naar een **gateway** (zoals de Dragino LPS8v2), die de berichten via ethernet, wifi of 4G doorstuurt naar het internet. Dit kan via een eigen gateway of een publiek LoRaWAN-netwerk (zoals The Things Network of Proximus MyThings).

**Alternatief:**
De **S31-NB** (NB-IoT) en **S31-CB** (NB-IoT + LTE-M) sturen berichten rechtstreeks via het mobiele netwerk (4G) met een simkaart per sensor. Zie stap 4.

### Opdracht 1.1
Bepaal op basis van de casus je keuze (LoRaWAN of NB-IoT, -LB of -LS) en som op welke hardware of diensten je naast de drie sensoren nodig hebt.

<details>
<summary><b>Oplossing tonen</b></summary>

Een verdedigbare keuze: **3 × S31-LB (LoRaWAN, batterij)** à € 52,95 plus **één gateway** (bv. Dragino LPS8v2) die via de wifi op het erf aan het internet hangt. Verder nodig: een gratis TTN-account (netwerkserver), de decoder van Dragino, en een dashboard (Datacake of ThingsBoard hebben gratis instapniveaus). Totaal eenmalig ± € 160 sensoren + gateway, geen maandelijkse kosten.

Alternatief: **3 × S31-CB (NB-IoT/LTE-M)**, geen gateway, maar drie simkaarten met abonnement (enkele euro's per maand per sensor) en je moet een server/broker kiezen. Interessant als er geen wifi of stroom voor een gateway is, of bij maar één sensor.

-LS (zonnepaneel) is in een donkere stal zinloos; -LB (batterij) volstaat: bij 10 min interval meerdere jaren (handleiding §4).

</details>

## Stap 2 · De handleiding: Dragino

Dragino publiceert alle documentatie online: <https://wiki.dragino.com/docs/LoRaWAN-End-Node/> ▸ *Temperature & Humidity Sensors* ▸ **S31/S31B-LB / LS**:
<https://wiki.dragino.com/docs/LoRaWAN-End-Node/temperature-humidity-sensors/s31-lb/>

Relevante secties voor configuratie en uitlezing:

| Sectie | Titel | Inhoud |
|---|---|---|
| **1.3** | Specification | Meetbereik (−40…80 °C), nauwkeurigheid en spanning. |
| **1.4** | Sleep mode and working mode | Energiebeheer: de sensor slaapt tussen metingen om batterij te sparen. |
| **1.6** | BLE connection | Configuratie via **Bluetooth** met een smartphone-app. |
| **2.2** | Quick guide to connect to LoRaWAN server (OTAA) | Stappen om het toestel aan te melden op TTN (DevEUI, AppKey, join-procedure). |
| **2.3** | Uplink Payload | Structuur van het 11-byte meetbericht (FPORT=2). |
| **2.4** | Payload Decoder file | Verwijzing naar de officiële JavaScript-decoder. |
| **2.6** | Temperature Alarm Feature | Instellingen voor alarmberichten bij temperatuuroverschrijding. |
| **3** | Configure (AT-commando's) | `AT+TDC` (zendinterval), `AT+SHTEMP` (alarmdrempel). |
| **4** | Battery & Power Consumption | Verwachte batterijduur per zendinterval. |

### Checklist voor ingebruikname van een LoRaWAN-sensor

- [ ] **Frequentieband:** EU868
- [ ] **Sleutels:** DevEUI, JoinEUI en AppKey (op het toestel of de verpakking)
- [ ] **Netwerkserver:** The Things Network (TTN) of ChirpStack
- [ ] **Gateway:** Actief en verbonden met het internet
- [ ] **Payload-decoder:** JavaScript-decoder van de fabrikant
- [ ] **Bestemming:** Dashboard (bv. Datacake, ThingsBoard) of integratie via MQTT/HTTP

### Opdracht 2.1
Raadpleeg §2.2 van de handleiding en noteer het gedrag van de groene led bij een geslaagde *join*. Zoek in §3 het AT-commando om het zendinterval op 10 minuten (600 seconden) in te stellen.

<details>
<summary><b>Oplossing tonen</b></summary>

§2.2 stap 3: na het indrukken van de knop (5 s) knippert de groene led 5× snel, het toestel probeert te joinen, en **na een geslaagde join brandt de groene led 5 s continu**. Daarna begint het uplinks te sturen.

§3.3.1 *Set Transmit Interval Time*: `AT+TDC=600` (waarde in seconden; standaard 1200 = 20 min). Hetzelfde kan via een downlink vanuit TTN: `01 00 02 58` (0x000258 = 600).

</details>

## Stap 3 · De payload met de hand decoderen

Een LoRaWAN-bericht is compact (11 bytes) en bestaat uit ruwe waarden. Sectie 2.3.2 van de handleiding beschrijft de opbouw van een gewone meting (**FPORT = 2**):

| Bytes | 0–1 | 2–5 | 6 | 7–8 | 9–10 |
|---|---|---|---|---|---|
| **Inhoud** | Batterij (mV) | Unix-timestamp (s) | Alarmvlag, modus & PA8 | Temperatuur (×10, signed) | RV (×10) |

**Basisprincipes voor het uitlezen:**

1. **Hexadecimaal:** 1 byte = 8 bits = 2 hex-tekens (`00`…`FF`, oftewel 0…255).
2. **Big-endian:** Bij meerdere bytes staat de meest betekenisvolle byte vooraan (`0B45` = `0x0B` × 256 + `0x45` = 2885).
3. **Two's complement:** Voor negatieve temperaturen trek je 65536 af als het hoogste bit 1 is (`FF3F` = 65343 − 65536 = −193 → **−19,3 °C**).

**Voorbeeld uit de handleiding:**
- Batterij: `0B45` → 2885 mV (2,885 V)
- Temperatuur: `0105` → 26,1 °C en `FF3F` → −19,3 °C
- Relatieve vochtigheid: `0197` → 40,7 %

We bouwen een testpayload op met deze waarden en decoderen ze stap voor stap.

In [ ]:
# De voorbeeldpayload (11 bytes = 22 hex-tekens), opgebouwd met de voorbeeldwaarden uit de handleiding
payload_hex = "0B456AD703600001050197"

payload = bytes.fromhex(payload_hex)         # hex-tekst -> bytes
print("Aantal bytes:", len(payload))
for i, b in enumerate(payload):
    print(f"byte {i:2d}: hex {b:02X}  = decimaal {b:3d}  = binair {b:08b}")

### 3a · Batterij (bytes 0–1)

In [ ]:
batterij_mv = payload[0] * 256 + payload[1]          # big-endian: eerste byte weegt 256× zwaarder
# hetzelfde met een ingebouwde functie:
batterij_mv_2 = int.from_bytes(payload[0:2], byteorder="big")

print(f"bytes 0-1 = {payload[0]:02X} {payload[1]:02X} -> {batterij_mv} mV = {batterij_mv/1000:.3f} V")
assert batterij_mv == batterij_mv_2 == 2885           # voorbeeld uit de handleiding

### 3b · Tijdstip (bytes 2–5)

Een *Unix-timestamp* is het aantal seconden sinds 1 januari 1970, 00:00 UTC. De sensor heeft een interne klok die bij de *join* door de netwerkserver wordt gelijkgezet.

In [ ]:
from datetime import datetime, timezone

timestamp = int.from_bytes(payload[2:6], byteorder="big")
tijd_utc = datetime.fromtimestamp(timestamp, tz=timezone.utc)
print(f"bytes 2-5 = {payload[2:6].hex().upper()} -> {timestamp} s -> {tijd_utc:%Y-%m-%d %H:%M:%S} UTC")

### 3c · Vlaggenbyte (byte 6)

Byte 6 bevat drie afzonderlijke statuswaarden op afzonderlijke bits:

| Bit 7 | Bits 6–2 | Bit 1 | Bit 0 |
|---|---|---|---|
| Status PA8 (1 = laag) | Modus (0 = normale meting, 31 = poll-antwoord) | — | Alarmvlag (1 = alarm) |

Je leest afzonderlijke bits uit met een bitmasker (`&`) en een bitverschuiving (`>>`).

In [ ]:
vlaggen = payload[6]
alarm   = vlaggen & 0x01            # masker 0000 0001
modus   = (vlaggen & 0x7C) >> 2     # masker 0111 1100, daarna 2 posities naar rechts
pa8     = (vlaggen & 0x80) >> 7     # masker 1000 0000

print(f"byte 6 = {vlaggen:02X} = {vlaggen:08b}")
print("alarm :", "JA" if alarm else "nee")
print("modus :", modus, "(0 = gewone meting)")
print("PA8   :", "laag" if pa8 else "hoog")

### 3d · Temperatuur (bytes 7–8) — let op het teken

In [ ]:
ruw = int.from_bytes(payload[7:9], byteorder="big")
if ruw & 0x8000:                     # hoogste bit gezet -> negatief getal
    ruw -= 65536
temperatuur = ruw / 10
print(f"bytes 7-8 = {payload[7:9].hex().upper()} -> {ruw} -> {temperatuur} °C")

# Python kan dit ook rechtstreeks met signed=True:
assert int.from_bytes(payload[7:9], "big", signed=True) / 10 == temperatuur

# Controleer het negatieve voorbeeld uit de handleiding: FF3F -> -19.3
print("FF3F ->", int.from_bytes(bytes.fromhex("FF3F"), "big", signed=True) / 10, "°C")

### 3e · Relatieve vochtigheid (bytes 9–10)

In [ ]:
vochtigheid = int.from_bytes(payload[9:11], byteorder="big") / 10
print(f"bytes 9-10 = {payload[9:11].hex().upper()} -> {vochtigheid} % RV")

### 3f · Alles samen in één functie

Een payload-decoder op de netwerkserver zet de ruwe bytes om naar leesbare variabelen:

In [ ]:
def decodeer_s31lb(payload_hex: str, fport: int = 2) -> dict:
    '''Decodeer een S31-LB-uplink (FPORT 2) volgens Dragino-handleiding sectie 2.3.2.'''
    b = bytes.fromhex(payload_hex)
    if fport != 2 or len(b) != 11:
        raise ValueError("Verwacht 11 bytes op FPORT 2")
    return {
        "batterij_V":   int.from_bytes(b[0:2], "big") / 1000,
        "tijd_utc":     datetime.fromtimestamp(int.from_bytes(b[2:6], "big"), tz=timezone.utc).isoformat(),
        "alarm":        bool(b[6] & 0x01),
        "modus":        (b[6] & 0x7C) >> 2,
        "temperatuur_C": int.from_bytes(b[7:9], "big", signed=True) / 10,
        "RV_pct":       int.from_bytes(b[9:11], "big") / 10,
    }

decodeer_s31lb(payload_hex)

### Opdracht 3.1 · Decodeer zelf

Hieronder staan drie meetberichten uit de TTN-console. Decodeer ze handmatig en controleer je uitkomst met de Python-functie.

| | Payload (hex) | Context |
|---|---|---|
| **A** | `0BC46AD703600000D60278` | Afdeling 3, normale meting |
| **B** | `0AE6696486D800FFCE0393` | Buitensensor in januari |
| **C** | `0B866A4671380101400320` | Afdeling 1, juli |

**Vragen:**
1. Wat is de temperatuur en relatieve vochtigheid in elk geval?
2. Welk bericht bevat een alarm en waarom?
3. Welke sensor geeft de laagste batterijspanning aan?

In [ ]:
# Opdracht 3.1 — controle
for naam, p in {"A": "0BC46AD703600000D60278", "B": "0AE6696486D800FFCE0393", "C": "0B866A4671380101400320"}.items():
    print(naam, decodeer_s31lb(p))

<details>
<summary><b>Oplossing tonen</b></summary>

| | Batterij | Tijd (UTC) | Alarm | T (°C) | RV (%) |
|---|---|---|---|---|---|
| A | 3,012 V | 2026-10-20 06:00 | nee | 21,4 | 63,2 |
| B | 2,790 V | 2026-01-12 05:30 | nee | **−5,0** (`FFCE` = 65486 → −50) | 91,5 |
| C | 2,950 V | 2026-07-02 14:10 | **ja** (byte 6 = `01`) | 32,0 | 80,0 |

C is een alarm-uplink: de temperatuur overschrijdt de ingestelde drempel (hittestress in juli); de sensor stuurt dan onmiddellijk, buiten het normale interval. B heeft de laagste batterijspanning (2,79 V), mede omdat een Li-SOCl₂-cel bij vrieskou tijdelijk een lagere spanning geeft.

Veelgemaakte fouten: little-endian lezen (`450B` i.p.v. `0B45`), het teken vergeten bij B, de vlaggenbyte decimaal lezen, niet door 10 delen.

</details>

## Stap 3g · Controle met de officiële Dragino-decoder

Dragino stelt officiële decoders beschikbaar op GitHub: <https://github.com/dragino/dragino-end-node-decoder>.
Dit JavaScript-bestand voeg je in TTN toe onder *Payload formatters ▸ Custom JavaScript*. We voeren het hier uit met Node.js om onze Python-functie te verifiëren.

In [ ]:
import requests, subprocess, json, shutil, textwrap

url_decoder = ("https://raw.githubusercontent.com/dragino/dragino-end-node-decoder/main/"
               "S31-LB%26S31B-LB/S31-LB_v1.3_TTN_decoder.txt")
js = requests.get(url_decoder, timeout=15).text
print(js[js.find("function Decoder"):][:900])     # toon het begin van de decoderfunctie

In [ ]:
if shutil.which("node") is None:
    print("Node.js niet gevonden — sla deze controle over of vergelijk de code handmatig.")
else:
    bytes_js = ", ".join(str(b) for b in bytes.fromhex(payload_hex))
    script = js + f"\nconsole.log(JSON.stringify(Decoder([{bytes_js}], 2)));\n"
    uit = subprocess.run(["node", "-e", script], capture_output=True, text=True, timeout=20)
    resultaat_js = json.loads(uit.stdout)
    print("Officiële decoder :", resultaat_js)
    print("Onze Python-functie:", decodeer_s31lb(payload_hex))
    assert resultaat_js["TempC_SHT31"] == decodeer_s31lb(payload_hex)["temperatuur_C"]
    assert resultaat_js["Hum_SHT31"]   == decodeer_s31lb(payload_hex)["RV_pct"]
    print("\nBeide geven dezelfde temperatuur en RV. (Data_time staat in de lokale tijdzone van Node.)")

## Stap 4 · De sensor via NB-IoT/LTE-M (S31-CB)

Handleiding S31-CB: <https://wiki.dragino.com/docs/NB-IoT/temperature-humidity-sensors/s31-cb/>

De NB-IoT-variant gebruikt een 4G-module (Quectel BG95) met een nanosimkaart. De sensor verbindt rechtstreeks met het internet en stuurt de metingen naar een gekozen server.

Met het AT-commando `AT+PRO` stel je het **protocol** en het **payloadformaat** in (§2.1–2.2):

```text
AT+PRO=1,0   // CoAP, hex-payload
AT+PRO=1,5   // CoAP, JSON-payload
AT+PRO=2,5   // UDP,  JSON
AT+PRO=3,0   // MQTT, hex
AT+PRO=3,5   // MQTT, JSON
AT+PRO=4,5   // TCP,  JSON
```

Voor MQTT stel je via AT-commando's het serveradres, het topic en het zendinterval in (`AT+SERVADDR`, `AT+PUBTOPIC`, `AT+TDC=600`, `AT+APN`).

**Vergelijking van de protocollen:**

| Protocol | Transport | Model | Kenmerken |
|---|---|---|---|
| **MQTT** | TCP | Publish/Subscribe via broker | Geschikt voor meerdere afnemers, dashboards en gegarandeerde aflevering (QoS). |
| **CoAP** | UDP | Request/Response | Laag energie- en dataverbruik ("HTTP voor IoT"). |
| **UDP** | UDP | Datagrammen | Minimale overhead, geen aflevergarantie. |
| **TCP** | TCP | Stream | Directe verbinding met aflevergarantie. |

In modus `...,5` verzendt de sensor gegevens direct in **JSON-formaat**. Dit maakt de payload groter, maar de gegevens zijn meteen leesbaar zonder decoder.

In [ ]:
import json, pandas as pd

voorbeeld_json = '''{"IMEI":"868508065628110","IMSI":"460240210507481","Model":"S31x-CB",
"temperature":25.6,"humidity":52.5,"interrupt":0,"interrupt_level":0,"count":0,
"battery":3.48,"signal":21,"time":"2024/12/03 03:24:54",
"latitude":0.000000,"longitude":0.000000,"gps_time":"1970-01-01T00:00:00Z",
"1":[24.2,52.5,"2024-12-02T05:34:19Z"],"2":[24.1,52.5,"2024-12-02T05:19:19Z"],
"3":[24.1,52.6,"2024-12-02T05:04:19Z"],"4":[24.2,52.8,"2024-12-02T04:49:19Z"],
"5":[24.3,52.0,"2024-12-02T04:34:19Z"],"6":[24.2,51.6,"2024-12-02T04:19:19Z"],
"7":[24.5,50.2,"2024-12-02T03:39:02Z"],"8":[24.3,50.1,"2024-12-02T03:24:02Z"]}'''

bericht = json.loads(voorbeeld_json)
print("Actuele meting:", bericht["temperature"], "°C,", bericht["humidity"], "% RV,",
      "batterij", bericht["battery"], "V, signaal", bericht["signal"])

# De sleutels "1".."8" zijn de laatste 8 metingen (datalog): [temperatuur, RV, tijdstip]
historiek = pd.DataFrame([bericht[str(i)] for i in range(1, 9)],
                         columns=["temperatuur_C", "RV_pct", "tijd"])
historiek["tijd"] = pd.to_datetime(historiek["tijd"])
historiek = historiek.sort_values("tijd")
historiek

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(8, 3.5))
ax1.plot(historiek["tijd"], historiek["temperatuur_C"], marker="o", label="temperatuur (°C)")
ax1.set_ylabel("°C")
ax2 = ax1.twinx()
ax2.plot(historiek["tijd"], historiek["RV_pct"], marker="s", color="tab:gray", label="RV (%)")
ax2.set_ylabel("% RV")
ax1.set_title("Datalog uit één S31-CB-uplink (voorbeeld uit de handleiding)")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

### Opdracht 4.1
Geef de AT-commando's om de S31-CB in te stellen voor MQTT (JSON-formaat), met een zendinterval van 10 minuten naar `broker.hivemq.com` op topic `ugent/i700301/stal/afdeling3`.

### Opdracht 4.2
De LoRaWAN-variant stuurt 11 bytes per bericht; de NB-IoT-variant in JSON-modus enkele honderden bytes. Leg uit waarom LoRaWAN zo compact moet zijn en waarom dit bij NB-IoT minder strikt is (denk aan de *duty cycle* op 868 MHz en de kostenstructuur).

<details>
<summary><b>Oplossing tonen</b></summary>

**4.1**
```
AT+PRO=3,5
AT+SERVADDR=broker.hivemq.com,1883
AT+CLIENT=s31cb-afdeling3
AT+PUBTOPIC=ugent/i700301/stal/afdeling3
AT+MQOS=1
AT+TDC=600
AT+APN=<apn van de operator>
```
Geen `AT+UNAME`/`AT+PWD`: de publieke broker vraagt geen aanmelding. Bij een eigen broker wel, plus `AT+TLSMOD` voor versleuteling.

**4.2** (1) Op 868 MHz geldt een wettelijke *duty cycle* van 1 %: hoogstens 36 s zendtijd per uur, en met 0,3–50 kbit/s kost elke byte veel zendtijd; TTN hanteert bovendien een fair-use-beleid van 30 s uplink per dag. (2) Zenden kost energie: 11 bytes = milliseconden, dus jaren op één batterij. NB-IoT gebruikt gelicentieerd spectrum zonder duty cycle, haalt tientallen kbit/s, en je betaalt per simkaart per maand (binnen een bundel), niet per byte: 300 bytes JSON zijn dan geen probleem, al kost het wel meer batterij.

</details>

## Overzicht en vervolg

| Onderwerp | Onderdeel in dit notebook |
|---|---|
| **Verbindingstypes** | LoRaWAN (S31-LB), Bluetooth (configuratie), Ethernet/Wifi/4G (gateway), NB-IoT/LTE-M (S31-CB) |
| **Protocollen** | MQTT, CoAP, UDP, TCP (kiesbaar via `AT+PRO`); HTTP (via netwerkserver integraties) |

In `02_mqtt_live_meekijken.ipynb` werken we verder met een virtuele S31-CB die live JSON-berichten publiceert.